In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
import re
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from urllib.parse import urlparse

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class ModelConfig:
    """Configuration for model paths and hyperparameters"""
    url_model_path: str = r"C:\Users\angelo\Downloads\THESIS\URL_Expert-20251210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl"
    text_model_path: str = r"C:\Users\angelo\Downloads\THESIS\distilbert_phishing_model"
    gating_network_path: str = "gating_network.pth"
    max_text_length: int = 128
    phishing_threshold: float = 0.5
    confidence_threshold: float = 0.7  # For high-confidence predictions

# ============================================================================
# ENHANCED URL FEATURES
# ============================================================================

class URLFeatures(BaseEstimator, TransformerMixin):
    """
    Enhanced feature extractor for URL-based phishing detection.
    Extracts structural and semantic characteristics from URLs.
    """
    
    SUSPICIOUS_TLDS = {'.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.click'}
    LEGITIMATE_DOMAINS = {'google.com', 'microsoft.com', 'apple.com', 'amazon.com', 
                          'facebook.com', 'paypal.com', 'github.com'}
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, urls):
        urls = np.array(urls).reshape(-1)
        feats = np.array([self._extract_features(u) for u in urls])
        return csr_matrix(feats)
    
    def _extract_features(self, url: str) -> List[float]:
        """Extract comprehensive URL features"""
        if not url or pd.isna(url):
            return [0] * 8  # Match original feature count
        
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            
            return [
                len(url),                                    # Total URL length
                url.count('-'),                              # Hyphen count
                url.count('@'),                              # @ symbol (credential spoofing)
                url.count('?'),                              # Query parameters
                url.count('='),                              # Parameter assignments
                url.count('.'),                              # Subdomain/path segments
                int(url.startswith("https")),                # HTTPS protocol
                int(url.count("//") > 1),                    # Multiple slashes
            ]
        except Exception:
            return [0] * 8
    
    @staticmethod
    def _calculate_entropy(text: str) -> float:
        """Calculate Shannon entropy of domain"""
        if not text:
            return 0.0
        prob = [text.count(c) / len(text) for c in set(text)]
        return -sum(p * np.log2(p) for p in prob if p > 0)

# ============================================================================
# ENHANCED GATING NETWORK
# ============================================================================

class GatingNetwork(nn.Module):
    """
    Enhanced gating network compatible with original trained weights.
    Learns optimal weight distribution between URL and Text experts.
    """
    def __init__(self, input_size=8, hidden_size=64, num_experts=2, use_batchnorm=False, dropout=0.0):
        super(GatingNetwork, self).__init__()
        self.use_batchnorm = use_batchnorm
        
        self.fc1 = nn.Linear(input_size, hidden_size)
        
        if use_batchnorm:
            self.bn1 = nn.BatchNorm1d(hidden_size)
        
        self.relu = nn.ReLU()
        
        if dropout > 0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = None
            
        self.fc2 = nn.Linear(hidden_size, num_experts)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        x = self.fc1(x)
        
        if self.use_batchnorm and x.size(0) > 1:
            x = self.bn1(x)
            
        x = self.relu(x)
        
        if self.dropout is not None:
            x = self.dropout(x)
            
        x = self.fc2(x)
        weights = self.softmax(x)
        return weights

# ============================================================================
# MODEL LOADER
# ============================================================================

class ModelLoader:
    """Centralized model loading with error handling"""
    
    @staticmethod
    def load_models(config: ModelConfig) -> Tuple:
        """Load all required models"""
        print("Loading Expert Models...")
        print("-" * 70)
        
        try:
            # Expert 1: URL-based detector
            expert_1 = joblib.load(config.url_model_path)
            print("✓ Expert 1 (URL-based): Loaded successfully")
        except Exception as e:
            print(f"✗ Error loading URL expert: {e}")
            raise
        
        try:
            # Expert 2: Text-based detector
            tokenizer = AutoTokenizer.from_pretrained(config.text_model_path)
            expert_2 = AutoModelForSequenceClassification.from_pretrained(config.text_model_path)
            expert_2.eval()
            print("✓ Expert 2 (Text-based): Loaded successfully")
        except Exception as e:
            print(f"✗ Error loading Text expert: {e}")
            raise
        
        try:
            # Gating network - compatible with original architecture
            gating_net = GatingNetwork(input_size=8, hidden_size=64, num_experts=2, 
                                      use_batchnorm=False, dropout=0.0)
            gating_net.load_state_dict(torch.load(config.gating_network_path))
            gating_net.eval()
            print("✓ Gating Network: Loaded successfully")
        except Exception as e:
            print(f"✗ Error loading Gating Network: {e}")
            raise
        
        print("\n" + "=" * 70)
        print("MoE System Initialization Complete")
        print("=" * 70 + "\n")
        
        return expert_1, expert_2, tokenizer, gating_net

# ============================================================================
# ENHANCED FEATURE EXTRACTION
# ============================================================================

class FeatureExtractor:
    """Enhanced feature extraction with improved phishing indicators"""
    
    # Expanded and weighted phishing phrase dictionary
    PHRASE_DICT = {
        # Urgency indicators (high weight)
        'urgent': 0.5, 'immediately': 0.5, 'act now': 0.5, 'limited time': 0.4,
        'expires today': 0.6, 'last chance': 0.5, 'don\'t miss': 0.4,
        
        # Account-related (very high weight)
        'verify account': 0.7, 'suspended': 0.6, 'confirm your': 0.6,
        'update account': 0.6, 'security alert': 0.7, 'unusual activity': 0.6,
        'verify identity': 0.7, 'locked': 0.6, 'restricted': 0.5,
        
        # Financial incentives (high weight)
        'congratulations': 0.5, 'winner': 0.6, 'claim': 0.5, 'prize': 0.5,
        'free money': 0.6, 'cash prize': 0.6, 'refund': 0.4, 'bonus': 0.3,
        
        # Action requests (medium-high weight)
        'click here': 0.5, 'click now': 0.5, 'download': 0.3, 'open attachment': 0.4,
        'confirm': 0.3, 'validate': 0.4, 'reactivate': 0.5,
        
        # Generic spam (lower weight)
        'free': 0.2, 'offer': 0.2, 'deal': 0.2, 'discount': 0.2,
    }
    
    @staticmethod
    def preprocess_text(text: str) -> str:
        """Enhanced text preprocessing"""
        if pd.isna(text) or text == "":
            return ""
        
        # Remove URLs
        text = re.sub(r'http\S+|www\.\S+', '', text)
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Remove excessive punctuation but keep sentence structure
        text = re.sub(r'([!?.]){2,}', r'\1', text)
        
        return text
    
    @classmethod
    def calculate_phrase_score(cls, text: str) -> float:
        """Calculate weighted phishing phrase score"""
        if not text:
            return 0.0
        
        text_lower = text.lower()
        score = 0.0
        matches = 0
        
        for phrase, weight in cls.PHRASE_DICT.items():
            if phrase in text_lower:
                score += weight
                matches += 1
        
        # Normalize by number of matches to prevent over-scoring
        if matches > 0:
            score = score / (1 + np.log(matches))
        
        return min(score, 1.0)
    
    @staticmethod
    def extract_gating_features(text: str, url: str, phrase_score: float) -> np.ndarray:
        """
        Extract features for gating network with intelligent URL/Text routing.
        When only URL is present, features heavily favor URL expert.
        When only text is present, features favor text expert.
        """
        url_present = 1 if (url and not pd.isna(url) and url != "") else 0
        text_present = 1 if (text and text.strip()) else 0
        
        # Text-based features
        if text:
            words = text.split()
            message_length = len(words)
            special_char_count = len(re.findall(r'[^\w\s]', text))
            hashtag_count = text.count('#')
            url_count = len(re.findall(r'http\S+|www\.\S+', text))
            capital_ratio = sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0.0
        else:
            message_length = 0
            special_char_count = 0
            hashtag_count = 0
            url_count = 0
            capital_ratio = 0.0
        
        # Enhanced embedding summary to guide expert selection
        # Positive values favor URL expert, negative values favor text expert
        if url_present and not text_present:
            # URL only - strongly favor URL expert
            embedding_summary = 1.0
        elif text_present and not url_present:
            # Text only - strongly favor text expert
            embedding_summary = -1.0
        elif url_present and text_present:
            # Both present - let phrase score and text features decide
            # High phrase score suggests text-based phishing
            if phrase_score > 0.3:
                embedding_summary = -0.5  # Lean towards text expert
            else:
                embedding_summary = 0.5   # Lean towards URL expert
        else:
            # Neither present (shouldn't happen)
            embedding_summary = 0.0
        
        # Original 8 features to match trained gating network
        features = np.array([
            url_present,
            phrase_score,
            message_length,
            special_char_count,
            hashtag_count,
            url_count,
            capital_ratio,
            embedding_summary  # Now intelligently guides expert selection
        ], dtype=np.float32)
        
        return features

# ============================================================================
# PHISHING DETECTOR
# ============================================================================

class PhishingDetector:
    """Main phishing detection system using Mixture of Experts"""
    
    def __init__(self, config: ModelConfig):
        self.config = config
        self.expert_1, self.expert_2, self.tokenizer, self.gating_net = \
            ModelLoader.load_models(config)
        self.feature_extractor = FeatureExtractor()
    
    def predict(self, text: str, url: str = "") -> Dict:
        """
        Perform phishing detection with intelligent expert routing.
        
        URL-only inputs: Heavily weighted towards URL expert
        Text-only inputs: Heavily weighted towards text expert
        Combined inputs: Smart aggregation with veto power
        
        Returns:
            Dictionary containing prediction results with confidence metrics
        """
        # Preprocess inputs
        text = self.feature_extractor.preprocess_text(text)
        phrase_score = self.feature_extractor.calculate_phrase_score(text)
        
        # Determine input type for intelligent routing
        has_url = bool(url and url.strip())
        has_text = bool(text and text.strip())
        
        # Get expert predictions with error handling
        url_probs = self._get_url_prediction(url)
        text_probs = self._get_text_prediction(text)
        
        # Compute gating weights
        expert_weights = self._compute_gating_weights(text, url, phrase_score)
        
        # Apply intelligent weighting with veto rules
        url_is_phishing = url_probs[1] > 0.5
        text_is_phishing = text_probs[1] > 0.5
        url_confidence = max(url_probs)
        text_confidence = max(text_probs)
        
        if has_url and not has_text:
            # URL only - force high weight to URL expert
            expert_weights = np.array([0.95, 0.05])
            
        elif has_text and not has_url:
            # Text only - force high weight to text expert
            expert_weights = np.array([0.05, 0.95])
            
        elif has_url and has_text:
            # Both present - smart combination with veto power
            
            # VETO RULE 1: High-confidence URL phishing detection
            if url_is_phishing and url_confidence > 0.75:
                # URL expert detected phishing with high confidence
                # Don't let text expert override it
                expert_weights = np.array([0.80, 0.20])
                
            # VETO RULE 2: Both experts agree on phishing
            elif url_is_phishing and text_is_phishing:
                # Both say phishing - strong signal
                expert_weights = np.array([0.60, 0.40])
                
            # VETO RULE 3: High phrase score suggests text-based phishing
            elif phrase_score > 0.5 and text_is_phishing:
                # Strong phishing indicators in text
                expert_weights = np.array([0.30, 0.70])
                
            # VETO RULE 4: URL says phishing but text says safe
            elif url_is_phishing and not text_is_phishing:
                # URL detected phishing, text didn't
                # Trust URL expert more (phishing URLs often have benign text)
                if url_confidence > 0.6:
                    expert_weights = np.array([0.75, 0.25])
                else:
                    expert_weights = np.array([0.60, 0.40])
                    
            # VETO RULE 5: Text says phishing but URL says safe
            elif text_is_phishing and not url_is_phishing:
                # Text detected phishing, URL didn't
                # Could be text-based phishing with legitimate URL
                if text_confidence > 0.7:
                    expert_weights = np.array([0.25, 0.75])
                else:
                    expert_weights = np.array([0.40, 0.60])
            
            # Otherwise use the gating network's learned weights
            # (when both say safe or low confidence predictions)
        
        # Ensemble prediction with weighted combination
        final_probs = (expert_weights[0] * url_probs + 
                      expert_weights[1] * text_probs)
        
        # Additional safety check: If URL expert says phishing with very high confidence,
        # ensure final prediction respects this even after combination
        if has_url and url_is_phishing and url_confidence > 0.85:
            # Force phishing probability to be at least 0.7
            final_probs[1] = max(final_probs[1], 0.7)
            final_probs[0] = 1.0 - final_probs[1]
        
        # Determine prediction and confidence
        prediction = "PHISHING" if final_probs[1] > self.config.phishing_threshold else "SAFE"
        confidence = max(final_probs) * 100
        
        # Determine if high confidence
        is_high_confidence = max(final_probs) > self.config.confidence_threshold
        
        # Determine routing method
        if has_url and not has_text:
            routing_method = "URL-only (URL expert prioritized)"
        elif has_text and not has_url:
            routing_method = "Text-only (Text expert prioritized)"
        elif has_url and has_text:
            if url_is_phishing and url_confidence > 0.75:
                routing_method = "Combined (URL veto - high confidence phishing)"
            elif url_is_phishing and text_is_phishing:
                routing_method = "Combined (Both experts agree - PHISHING)"
            elif url_is_phishing and not text_is_phishing:
                routing_method = "Combined (URL phishing, text safe - URL prioritized)"
            elif text_is_phishing and not url_is_phishing:
                routing_method = "Combined (Text phishing, URL safe - context-based)"
            else:
                routing_method = "Combined (Gating network)"
        else:
            routing_method = "Combined (Gating network)"
        
        return {
            'prediction': prediction,
            'confidence': confidence,
            'is_high_confidence': is_high_confidence,
            'url_weight': expert_weights[0] * 100,
            'text_weight': expert_weights[1] * 100,
            'url_prediction': 'PHISHING' if url_is_phishing else 'SAFE',
            'text_prediction': 'PHISHING' if text_is_phishing else 'SAFE',
            'url_confidence': url_confidence * 100,
            'text_confidence': text_confidence * 100,
            'phrase_score': phrase_score,
            'expert_agreement': url_is_phishing == text_is_phishing,
            'routing_method': routing_method,
            'input_type': 'URL+Text' if (has_url and has_text) else ('URL' if has_url else 'Text')
        }
    
    def _get_url_prediction(self, url: str) -> np.ndarray:
        """Get URL expert prediction with fallback"""
        if url and url.strip():
            try:
                url_df = pd.DataFrame({'url': [url]})
                return self.expert_1.predict_proba(url_df)[0]
            except Exception as e:
                print(f"Warning: URL prediction failed - {e}")
        return np.array([0.5, 0.5])
    
    def _get_text_prediction(self, text: str) -> np.ndarray:
        """Get text expert prediction with fallback"""
        if text:
            try:
                inputs = self.tokenizer(
                    text, 
                    return_tensors='pt', 
                    padding=True,
                    truncation=True, 
                    max_length=self.config.max_text_length
                )
                with torch.no_grad():
                    outputs = self.expert_2(**inputs)
                    return torch.softmax(outputs.logits, dim=1)[0].numpy()
            except Exception as e:
                print(f"Warning: Text prediction failed - {e}")
        return np.array([0.5, 0.5])
    
    def _compute_gating_weights(self, text: str, url: str, phrase_score: float) -> np.ndarray:
        """Compute expert weights using gating network"""
        gating_features = self.feature_extractor.extract_gating_features(
            text, url, phrase_score
        )
        gating_input = torch.FloatTensor(gating_features).unsqueeze(0)
        
        with torch.no_grad():
            weights = self.gating_net(gating_input)
        
        return weights[0].numpy()

# ============================================================================
# TESTING INTERFACE
# ============================================================================

def display_results(results: Dict, text: str = "", url: str = ""):
    """Display prediction results in formatted output"""
    print("=" * 70)
    print("PREDICTION RESULTS - Enhanced MoE Phishing Detection")
    print("=" * 70)
    
    # Display input information
    print(f"\nInput Type: {results['input_type']}")
    
    if text:
        display_text = text[:80] + "..." if len(text) > 80 else text
        print(f"Text: {display_text}")
    if url:
        print(f"URL: {url}")
    
    print("\n" + "-" * 70)
    print(f"Routing Method: {results['routing_method']}")
    print("\nExpert Weights:")
    print(f"  URL Expert:  {results['url_weight']:.2f}%")
    print(f"  Text Expert: {results['text_weight']:.2f}%")
    
    # Visual indicator of which expert is dominant
    url_dominant = results['url_weight'] > results['text_weight']
    print(f"\n  → Primary Expert: {'URL' if url_dominant else 'Text'}")
    
    print("\nIndividual Expert Predictions:")
    print(f"  URL Expert:  {results['url_prediction']} "
          f"(Confidence: {results['url_confidence']:.2f}%)")
    print(f"  Text Expert: {results['text_prediction']} "
          f"(Confidence: {results['text_confidence']:.2f}%)")
    
    print("\nAdditional Metrics:")
    print(f"  Phrase Score: {results['phrase_score']:.3f}")
    print(f"  Expert Agreement: {'Yes' if results['expert_agreement'] else 'No'}")
    
    print("-" * 70)
    
    # Color-coded confidence indicator
    confidence_level = "HIGH" if results['is_high_confidence'] else "MODERATE"
    print(f"FINAL PREDICTION: {results['prediction']}")
    print(f"Confidence: {results['confidence']:.2f}% ({confidence_level})")
    
    print("=" * 70)
    print()

def test_sample(detector: PhishingDetector, input_text: str) -> Dict:
    """Test the detector with automatic URL extraction"""
    # Extract URL if present
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    urls = re.findall(url_pattern, input_text)
    
    url = urls[0] if urls else ""
    text = re.sub(url_pattern, '', input_text).strip()
    
    # Get prediction
    results = detector.predict(text, url)
    
    # Display results
    display_results(results, text, url)
    
    return results

def interactive_mode(detector: PhishingDetector):
    """Interactive testing interface"""
    print("\n" + "=" * 70)
    print("INTERACTIVE MODE - Enhanced Phishing Detection")
    print("=" * 70)
    print("\nCommands:")
    print("  - Enter message/URL to analyze")
    print("  - 'exit' or 'quit' to end")
    print("  - 'sample' for predefined examples")
    print("  - 'batch' to test multiple samples")
    print("=" * 70)
    
    samples = [
        "URGENT! Your account has been suspended. Verify now at http://fake-bank.com",
        "Congratulations! You won $1000! Claim here: http://prize-claim.tk",
        "Hey, are we still meeting for lunch tomorrow?",
        "http://paypa1-secure-login.com/verify",
        "Please review the attached document and send feedback.",
        "SECURITY ALERT: Click here to confirm your identity immediately!",
        "Meeting notes from today's discussion are available on Drive.",
        "Limited time offer! Act now to claim your bonus!",
        "Your package delivery failed. Update address: http://fedex-track.xyz",
        "Thanks for the presentation yesterday. Looking forward to next steps.",
        # Additional URL-only test cases
        "http://secure-banking-update.tk",
        "https://amaz0n-account-verify.com/update",
        "http://192.168.1.1/admin",
        "http://paypal-security-alert.xyz/login",
        "https://www.google.com",
        "https://github.com/anthropics/claude",
    ]
    
    while True:
        print("\n" + "-" * 70)
        user_input = input("\nEnter command or message: ").strip()
        
        if user_input.lower() in ['exit', 'quit', 'q']:
            print("\n" + "=" * 70)
            print("SESSION ENDED")
            print("=" * 70 + "\n")
            break
        
        if user_input.lower() == 'batch':
            print("\nTesting all predefined samples...\n")
            for i, sample in enumerate(samples, 1):
                print(f"\n{'='*70}")
                print(f"Sample {i}/{len(samples)}")
                print(f"{'='*70}")
                test_sample(detector, sample)
            continue
        
        if user_input.lower() == 'sample':
            print("\nPredefined Samples:")
            for i, sample in enumerate(samples, 1):
                preview = sample[:60] + "..." if len(sample) > 60 else sample
                print(f"{i}. {preview}")
            
            try:
                choice = int(input("\nSelect sample (1-{}): ".format(len(samples)))) - 1
                if 0 <= choice < len(samples):
                    user_input = samples[choice]
                else:
                    print("Invalid selection")
                    continue
            except ValueError:
                print("Invalid input")
                continue
        
        if not user_input:
            print("No input provided")
            continue
        
        try:
            test_sample(detector, user_input)
        except Exception as e:
            print(f"\nError: {str(e)}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Initialize configuration and detector
    config = ModelConfig()
    detector = PhishingDetector(config)
    
    print("\n" + "=" * 70)
    print("SYSTEM READY")
    print("=" * 70)
    print("\nUsage:")
    print('  detector.predict("your text", "http://url")  - Single prediction')
    print('  test_sample(detector, "message")             - Test with display')
    print('  interactive_mode(detector)                   - Interactive session')
    print("\nQuick Start:")
    print('  >>> interactive_mode(detector)')
    print("=" * 70)

Loading Expert Models...
----------------------------------------------------------------------
✓ Expert 1 (URL-based): Loaded successfully
✓ Expert 2 (Text-based): Loaded successfully
✓ Gating Network: Loaded successfully

MoE System Initialization Complete


SYSTEM READY

Usage:
  detector.predict("your text", "http://url")  - Single prediction
  test_sample(detector, "message")             - Test with display
  interactive_mode(detector)                   - Interactive session

Quick Start:
  >>> interactive_mode(detector)


In [2]:
test_sample(detector,"Good afternoon Dean")

PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: Text
Text: Good afternoon Dean

----------------------------------------------------------------------
Routing Method: Text-only (Text expert prioritized)

Expert Weights:
  URL Expert:  5.00%
  Text Expert: 95.00%

  → Primary Expert: Text

Individual Expert Predictions:
  URL Expert:  SAFE (Confidence: 50.00%)
  Text Expert: SAFE (Confidence: 100.00%)

Additional Metrics:
  Phrase Score: 0.000
  Expert Agreement: Yes
----------------------------------------------------------------------
FINAL PREDICTION: SAFE
Confidence: 97.50% (HIGH)



{'prediction': 'SAFE',
 'confidence': 97.49969482421875,
 'is_high_confidence': True,
 'url_weight': 5.0,
 'text_weight': 95.0,
 'url_prediction': 'SAFE',
 'text_prediction': 'SAFE',
 'url_confidence': 50.0,
 'text_confidence': 99.99967813491821,
 'phrase_score': 0.0,
 'expert_agreement': True,
 'routing_method': 'Text-only (Text expert prioritized)',
 'input_type': 'Text'}

In [ ]:
 interactive_mode(detector)


INTERACTIVE MODE - Enhanced Phishing Detection

Commands:
  - Enter message/URL to analyze
  - 'exit' or 'quit' to end
  - 'sample' for predefined examples
  - 'batch' to test multiple samples

----------------------------------------------------------------------



Enter command or message:  please login to claim your free 1000 peso gcash at http://ledger-comstart.pages.dev/


PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: URL+Text
Text: please login to claim your free 1000 peso gcash at
URL: http://ledger-comstart.pages.dev/

----------------------------------------------------------------------
Routing Method: Combined (URL veto - high confidence phishing)

Expert Weights:
  URL Expert:  80.00%
  Text Expert: 20.00%

  → Primary Expert: URL

Individual Expert Predictions:
  URL Expert:  PHISHING (Confidence: 99.97%)
  Text Expert: PHISHING (Confidence: 96.91%)

Additional Metrics:
  Phrase Score: 0.413
  Expert Agreement: Yes
----------------------------------------------------------------------
FINAL PREDICTION: PHISHING
Confidence: 99.36% (HIGH)


----------------------------------------------------------------------



Enter command or message:  good afternoon this is the report of group 1


PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: Text
Text: good afternoon this is the report of group 1

----------------------------------------------------------------------
Routing Method: Text-only (Text expert prioritized)

Expert Weights:
  URL Expert:  5.00%
  Text Expert: 95.00%

  → Primary Expert: Text

Individual Expert Predictions:
  URL Expert:  SAFE (Confidence: 50.00%)
  Text Expert: SAFE (Confidence: 100.00%)

Additional Metrics:
  Phrase Score: 0.000
  Expert Agreement: Yes
----------------------------------------------------------------------
FINAL PREDICTION: SAFE
Confidence: 97.50% (HIGH)


----------------------------------------------------------------------


...

PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: Text
Text: LOOK: Alex Eala waves to the fans as she leaves the court after her women’s sing...

----------------------------------------------------------------------
Routing Method: Text-only (Text expert prioritized)

Expert Weights:
  URL Expert:  5.00%
  Text Expert: 95.00%

  → Primary Expert: Text

Individual Expert Predictions:
  URL Expert:  SAFE (Confidence: 50.00%)
  Text Expert: SAFE (Confidence: 99.10%)

Additional Metrics:
  Phrase Score: 0.000
  Expert Agreement: Yes
----------------------------------------------------------------------
FINAL PREDICTION: SAFE
Confidence: 96.64% (HIGH)


----------------------------------------------------------------------



Enter command or message:  𝐀𝐬𝐮𝐬 𝐑𝐎𝐆 𝐒𝐭𝐫𝐢𝐱 𝐒𝐂𝐀𝐑 𝐆𝟏𝟖 𝐆𝐚𝐦𝐢𝐧𝐠 𝐋𝐚𝐩𝐭𝐨𝐩 𝟏𝟏𝟎,𝟎𝟎𝟎 𝐎𝐍𝐋𝐘! 𝟏𝟏𝟎,𝟎𝟎𝟎 𝐎𝐍𝐋𝐘! 𝟏𝟏𝟎,𝟎𝟎𝟎 𝐎𝐍𝐋𝐘! ✅NVIDIA GeForce RTX 4080 12GB GDDR6 ✅Intel® Core™ i9-13980HX Processor(24cores, 32threads)  ✅32GB DDR5 RAM ✅1TB M.2 NVMe PCIe 4.0 SSD ✅18" ROG Nebula QHD+ 240Hz IPS 3ms G-Sync Display ✅RGB Backlit Keyboard ✅Windows 11 ✅Warranty Until November 2026 Inclusions: Laptop Charger ROG Bag Box MOP: Cash/Gcash Bank Transfer Credit Card Meetup within Metro Manila Pickup at our office Fairview QC Message for Inquiries!


PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: Text
Text: 𝐀𝐬𝐮𝐬 𝐑𝐎𝐆 𝐒𝐭𝐫𝐢𝐱 𝐒𝐂𝐀𝐑 𝐆𝟏𝟖 𝐆𝐚𝐦𝐢𝐧𝐠 𝐋𝐚𝐩𝐭𝐨𝐩 𝟏𝟏𝟎,𝟎𝟎𝟎 𝐎𝐍𝐋𝐘! 𝟏𝟏𝟎,𝟎𝟎𝟎 𝐎𝐍𝐋𝐘! 𝟏𝟏𝟎,𝟎𝟎𝟎 𝐎𝐍𝐋𝐘! ...

----------------------------------------------------------------------
Routing Method: Text-only (Text expert prioritized)

Expert Weights:
  URL Expert:  5.00%
  Text Expert: 95.00%

  → Primary Expert: Text

Individual Expert Predictions:
  URL Expert:  SAFE (Confidence: 50.00%)
  Text Expert: PHISHING (Confidence: 99.52%)

Additional Metrics:
  Phrase Score: 0.000
  Expert Agreement: No
----------------------------------------------------------------------
FINAL PREDICTION: PHISHING
Confidence: 97.04% (HIGH)


----------------------------------------------------------------------



Enter command or message:  DOST-PHIVOLCS and Subic-Clark Alliance for Development forged a partnership through signing a Memorandum of Agreement (MoA) on Integrating GeoRiskPH Platform with the planning and development of Subic-Clark Corridor held on January 16 at the PHIVOLCS Auditorium, Quezon City. Dr. Teresito C. Bacolcol, Director of DOST-PHIVOLCS along with Atty. Carminda Z. Fabros, Executive Director of Subic-Clark Alliance for Development led the ceremony witnessed by Mr. Eric P. Santos, Information Technology Officer II of DOST-PHIVOLCS and Atty. Aliw V. Del Rosario, Legal Counsel of Subic-Clark Alliance for Development. The partnership supports the institute’s cause in strengthening stakeholders’ disaster preparedness by using the centralized information resource of GeoRiskPH Platform for hazards assessment and decision-making. Also, Dr. Bacolcol toured the visitors around the data receiving center of DOST-PHIVOLCS.


PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: Text
Text: DOST-PHIVOLCS and Subic-Clark Alliance for Development forged a partnership thro...

----------------------------------------------------------------------
Routing Method: Text-only (Text expert prioritized)

Expert Weights:
  URL Expert:  5.00%
  Text Expert: 95.00%

  → Primary Expert: Text

Individual Expert Predictions:
  URL Expert:  SAFE (Confidence: 50.00%)
  Text Expert: SAFE (Confidence: 100.00%)

Additional Metrics:
  Phrase Score: 0.000
  Expert Agreement: Yes
----------------------------------------------------------------------
FINAL PREDICTION: SAFE
Confidence: 97.50% (HIGH)


----------------------------------------------------------------------
